In [1]:
import pandas as pd
import numpy as np

In [2]:
FILE_NAME = "2024-v210-06052026-EU MRV Publication of information.xlsx"
DATA_FOLDER = "examples/data/raw/"
PATH = DATA_FOLDER + FILE_NAME

In [3]:
df = pd.read_excel(PATH)

/home/eduard/anaconda3/envs/vessels-anal-back/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [8]:
df.columns

Index(['Ship', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Company', 'Unnamed: 9',
       ...
       'Unnamed: 103', 'Unnamed: 104', 'Unnamed: 105', 'Unnamed: 106',
       'Unnamed: 107', 'Unnamed: 108', 'Unnamed: 109', 'Unnamed: 110',
       'Unnamed: 111', 'Unnamed: 112'],
      dtype='str', length=113)

In [3]:
import os
import re
from typing import Iterable, Tuple

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 50)

RAW_OUT_DIR = "examples/data/interim/"
os.makedirs(RAW_OUT_DIR, exist_ok=True)

In [ ]:
# Read the sheet as a raw grid (no headers) to inspect the real header structure.
raw = pd.read_excel(PATH, header=None)
raw.iloc[:12, :25]

/home/eduard/anaconda3/envs/vessels-anal-back/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,Ship,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Company,NaN,DoC,NaN,Verifier,NaN,NaN,NaN,NaN,NaN,Monitoring methods,NaN,NaN,NaN,NaN,Annual monitoring results,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fuel consumption,NaN
2,IMO Number,Name,Ship type,Reporting Period,Technical efficiency,Port of Registry,Home Port,Ice Class,IMO Number,Name,DoC issue date,DoC expiry date,Verifier Name,Verifier Address,Verifier City,Verifier Country,Verifier Accreditation number,Verifier NAB,A,B,C,D,D,Total fuel consumption [m tonnes],Total fuel benefitting from a derogation in ac...
3,1013664,AFRICAN PEACOCK,Bulk carrier,2024,EEDI (4.12 gCO₂/t·nm),Panama,Panama,NaN,5112991,CSL MARITIME S.A.-TOKYO BRANCH,13/03/2025,30/06/2026,Nippon Kaiji Kyokai,"4-7, Kioi-Cho, Chiyoda-Ku","Tokyo, 102-8567",Japan,VS-21325-01,Deutsche Akkredierungsstelle GmbH,NaN,NaN,NaN,NaN,NaN,0,NaN
4,1013676,AQUADONNA,Bulk carrier,2024,EEDI (3.3 gCO₂/t·nm),PANAMA,NaN,NaN,6409613,BLUEOCEAN SHIPMANAGEMENT INTERNATIONAL INC.,31/01/2025,30/06/2026,Nippon Kaiji Kyokai,"4-7, Kioi-Cho, Chiyoda-Ku","Tokyo, 102-8567",Japan,VS-21325-01,Deutsche Akkredierungsstelle GmbH,NaN,NaN,Yes,NaN,NaN,735.317,NaN
5,1014591,NEW ACACIA,Bulk carrier,2024,EEDI (3.86 gCO₂/t·nm),MAJURO,NaN,NaN,5949344,FIBER MARINE SHIP MANAGEMENT & TRADING CO.,09/03/2025,30/06/2026,Nippon Kaiji Kyokai,"4-7, Kioi-Cho, Chiyoda-Ku","Tokyo, 102-8567",Japan,VS-21325-01,Deutsche Akkredierungsstelle GmbH,Yes,NaN,NaN,NaN,NaN,905.218,NaN
6,1014606,SEA GOAT,Bulk carrier,2024,EEDI (3.88 gCO₂/t·nm),PANAMA,NaN,NaN,6431581,STELLAR SHIPMANAGEMENT INC.,06/06/2025,30/06/2026,Nippon Kaiji Kyokai,"4-7, Kioi-Cho, Chiyoda-Ku","Tokyo, 102-8567",Japan,VS-21325-01,Deutsche Akkredierungsstelle GmbH,Yes,Yes,Yes,NaN,NaN,928.971,NaN
7,1014618,BIRD OF PARADISE,Bulk carrier,2024,EEDI (3.91 gCO₂/t·nm),PANAMA,NaN,NaN,6431581,STELLAR SHIPMANAGEMENT INC.,04/06/2025,30/06/2026,Nippon Kaiji Kyokai,"4-7, Kioi-Cho, Chiyoda-Ku","Tokyo, 102-8567",Japan,VS-21325-01,Deutsche Akkredierungsstelle GmbH,Yes,NaN,Yes,NaN,NaN,251.85,NaN
8,1014838,WARRIOR,Bulk carrier,2024,EEXI (3.99 gCO₂/t·nm),Monrovia,NaN,NaN,1069901,DryDel Shipping Inc.,30/03/2025,30/06/2026,HELLENIC LLOYD'S S.A.,348 Syggrou Avenue,Athens,Greece,1190,HELLENIC ACCREDITATION SYSTEM,Yes,NaN,NaN,NaN,NaN,1195.54,NaN
9,1014840,ALLEGRA,Bulk carrier,2024,EEDI (3.99 gCO₂/t·nm),Monrovia,NaN,NaN,1069901,DryDel Shipping Inc.,DoC not issued,DoC not issued,HELLENIC LLOYD'S S.A.,348 Syggrou Avenue,Athens,Greece,1190,HELLENIC ACCREDITATION SYSTEM,Yes,NaN,NaN,NaN,NaN,656.85,NaN


In [ ]:
# Heuristic: the first non-empty data row often starts where the IMO column becomes numeric-ish.
# We'll locate the first row with at least N numeric-ish values.

def _numericish(x) -> bool:
    if pd.isna(x):
        return False
    if isinstance(x, (int, float, np.integer, np.floating)):
        return True
    s = str(x).strip()
    return bool(re.fullmatch(r"\d{6,10}", s))

row_scores = raw.apply(lambda r: sum(_numericish(v) for v in r.values), axis=1)
row_scores.head(30), row_scores.idxmax()

(0      0
 1      0
 2      0
 3     33
 4     36
 5     36
 6     39
 7     45
 8     39
 9     39
 10    39
 11    39
 12    40
 13    40
 14    39
 15    36
 16    36
 17    36
 18    33
 19    36
 20    33
 21    33
 22    33
 23    33
 24    33
 25    48
 26    39
 27    48
 28    39
 29    48
 dtype: int64,
 3207)

In [ ]:
# Choose header rows (3-level headers) and the first data row.
# Adjust these two values if the preview above suggests different rows.
HEADER_ROWS = (0, 1, 2)
DATA_START_ROW = 3

hdr = raw.iloc[list(HEADER_ROWS)].copy()
data = raw.iloc[DATA_START_ROW:].copy()

hdr.shape, data.shape

((3, 113), (14132, 113))

In [ ]:
# Forward-fill header blocks horizontally so merged Excel cells become explicit.
hdr_ff = hdr.ffill(axis=1)

# Build a MultiIndex from the 3 header rows.
cols_mi = pd.MultiIndex.from_arrays([hdr_ff.iloc[i].astype(object).values for i in range(hdr_ff.shape[0])])

def _clean_header_token(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    if s.lower().startswith("unnamed:"):
        return ""
    return s

def flatten_multiindex(mi: pd.MultiIndex, sep: str = " | ") -> list[str]:
    out = []
    for tup in mi.tolist():
        parts = [_clean_header_token(t) for t in tup]
        parts = [p for p in parts if p]
        name = sep.join(parts) if parts else ""
        name = re.sub(r"\s+", " ", name).strip()
        out.append(name)

    # Make unique (stable) column names.
    counts = {}
    uniq = []
    for c in out:
        base = c or "col"
        n = counts.get(base, 0)
        counts[base] = n + 1
        uniq.append(base if n == 0 else f"{base}__{n+1}")
    return uniq

flat_cols = flatten_multiindex(cols_mi)
flat_cols[:20], len(flat_cols)

(['Ship | IMO Number',
  'Ship | Name',
  'Ship | Ship type',
  'Ship | Reporting Period',
  'Ship | Technical efficiency',
  'Ship | Port of Registry',
  'Ship | Home Port',
  'Ship | Ice Class',
  'Company | IMO Number',
  'Company | Name',
  'DoC | DoC issue date',
  'DoC | DoC expiry date',
  'Verifier | Verifier Name',
  'Verifier | Verifier Address',
  'Verifier | Verifier City',
  'Verifier | Verifier Country',
  'Verifier | Verifier Accreditation number',
  'Verifier | Verifier NAB',
  'Monitoring methods | A',
  'Monitoring methods | B'],
 113)

In [ ]:
df_raw = data.copy()
df_raw.columns = flat_cols

# Drop completely empty rows/cols.
df_raw = df_raw.dropna(axis=0, how="all").dropna(axis=1, how="all")

# Keep a raw snapshot *after* header normalization (useful for lineage/debugging).
raw_snapshot_path = os.path.join(RAW_OUT_DIR, "mrv_raw_flat.parquet")
df_raw.to_parquet(raw_snapshot_path, index=False)
raw_snapshot_path, df_raw.shape

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
# Cleaning helpers: normalize missing values, coerce numeric, and keep an audit of coercions.

def normalize_missing(s: pd.Series) -> pd.Series:
    if s.dtype == object:
        s2 = s.astype(str).str.strip()
        s2 = s2.replace({"": np.nan, "nan": np.nan, "NaN": np.nan, "None": np.nan, "N/A": np.nan, "n/a": np.nan})
        # Keep original objects where possible
        out = s.copy()
        out.loc[~s.isna()] = s2.loc[~s.isna()].values
        return out
    return s

NON_NUMERIC_SENTINELS = {
    "division by zero": np.nan,
    "Division by zero": np.nan,
    "DIVISION BY ZERO": np.nan,
    "#DIV/0!": np.nan,
    "#VALUE!": np.nan,
    "-": np.nan,
}

def coerce_numeric(series: pd.Series) -> Tuple[pd.Series, pd.DataFrame]:
    s = series.copy()
    if s.dtype == object:
        s = s.replace(NON_NUMERIC_SENTINELS)
        s = s.astype(str).str.replace(",", "", regex=False).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    num = pd.to_numeric(s, errors="coerce")
    audit = pd.DataFrame({
        "raw": series,
        "coerced": num,
        "was_coerced_to_nan": (~series.isna()) & (num.isna()),
    })
    return num, audit

# Example: detect candidate numeric columns by sampling.
sample = df_raw.head(200)
obj_cols = [c for c in df_raw.columns if df_raw[c].dtype == object]

numeric_candidates = []
for c in obj_cols:
    num = pd.to_numeric(sample[c].replace(NON_NUMERIC_SENTINELS), errors="coerce")
    ratio = num.notna().mean()
    if ratio >= 0.8:  # tweak threshold
        numeric_candidates.append((c, ratio))

numeric_candidates[:15], len(numeric_candidates)

([('Ship | Reporting Period', np.float64(1.0)),
  ('Annual monitoring results | Fuel consumption | Total fuel consumption [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | Total CO₂ emissions [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | CO₂ emissions from all voyages between ports under a MS jurisdiction [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | CO₂ emissions from all voyages which departed from ports under a MS jurisdiction [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | CO₂ emissions from all voyages to ports under a MS jurisdiction [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | CO₂ emissions which occurred within ports under a MS jurisdiction at berth [m tonnes]',
   np.float64(1.0)),
  ('Annual monitoring results | CO₂ Emissions | CO₂ emissions which occurred within ports under a MS jurisdiction [m tonnes]'

In [ ]:
df = df_raw.copy()

# Normalize missing values across object columns.
for c in df.columns:
    df[c] = normalize_missing(df[c])

# Coerce candidate numeric columns and collect audits for debugging.
audits = {}
for c, _ratio in numeric_candidates:
    coerced, audit = coerce_numeric(df[c])
    df[c] = coerced
    # Store only if there were suspicious values.
    if audit["was_coerced_to_nan"].any():
        audits[c] = audit[audit["was_coerced_to_nan"]].head(20)

# Peek at coercion issues (if any)
list(audits.keys())[:10], {k: len(v) for k, v in list(audits.items())[:5]}

(['Annual monitoring results | Average energy efficiency | Fuel consumption per distance [kg / n mile]',
  'Annual monitoring results | Average energy efficiency | CO₂ emissions per distance [kg CO₂ / n mile]',
  'Annual monitoring results | Average energy efficiency | CO₂eq emissions per distance [kg CO₂eq / n mile]'],
 {'Annual monitoring results | Average energy efficiency | Fuel consumption per distance [kg / n mile]': 20,
  'Annual monitoring results | Average energy efficiency | CO₂ emissions per distance [kg CO₂ / n mile]': 20,
  'Annual monitoring results | Average energy efficiency | CO₂eq emissions per distance [kg CO₂eq / n mile]': 20})

In [ ]:
# Example: pick stable IDs and create a "vessels" table view.
# (We'll refine schema-driven extraction next; for now it's a learning playground.)

def find_col(pattern: str) -> list[str]:
    rx = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if rx.search(c)]

imo_cols = find_col(r"\bIMO\b")
ship_name_cols = find_col(r"Ship name|Name of ship")
company_cols = find_col(r"Company")
verifier_cols = find_col(r"Verifier")

imo_cols[:5], ship_name_cols[:5], company_cols[:5], verifier_cols[:5]

(['Ship | IMO Number', 'Company | IMO Number'],
 [],
 ['Company | IMO Number', 'Company | Name'],
 ['Verifier | Verifier Name',
  'Verifier | Verifier Address',
  'Verifier | Verifier City',
  'Verifier | Verifier Country',
  'Verifier | Verifier Accreditation number'])

In [ ]:
# Build simple normalized views.
IMO_COL = imo_cols[0] if imo_cols else None
SHIP_NAME_COL = ship_name_cols[0] if ship_name_cols else None

if IMO_COL is None:
    raise ValueError("Couldn't auto-detect IMO column; adjust find_col patterns.")

vessels = df[[c for c in [IMO_COL, SHIP_NAME_COL] if c]].copy()
vessels = vessels.rename(columns={IMO_COL: "imo_number", SHIP_NAME_COL: "ship_name"})

# Clean IMO as string ID (some spreadsheets store it as float)
vessels["imo_number"] = vessels["imo_number"].astype("Int64").astype(str)

# Drop rows without IMO
vessels = vessels[vessels["imo_number"].notna() & (vessels["imo_number"] != "<NA>")]

vessels = vessels.drop_duplicates(subset=["imo_number"]).reset_index(drop=True)
vessels.head(10), vessels.shape

(  imo_number
 0    1013664
 1    1013676
 2    1014591
 3    1014606
 4    1014618
 5    1014838
 6    1014840
 7    1014979
 8    1015155
 9    1015313,
 (14132, 1))

In [ ]:
# A "raw fact" table still matters: keep the full cleaned frame for later feature engineering.
clean_snapshot_path = os.path.join(RAW_OUT_DIR, "mrv_cleaned.parquet")
df.to_parquet(clean_snapshot_path, index=False)
clean_snapshot_path, df.shape

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
# Gap filling (playground): choose strategy per column type.
# - Numeric: median (or domain-specific default)
# - Categorical/text: mode or "unknown"

filled = df.copy()

num_cols = filled.select_dtypes(include=["number"]).columns.tolist()
obj_cols = [c for c in filled.columns if c not in num_cols]

for c in num_cols:
    med = filled[c].median(skipna=True)
    if pd.notna(med):
        filled[c] = filled[c].fillna(med)

for c in obj_cols:
    # keep NaN if the column is mostly empty; otherwise fill with mode.
    non_na = filled[c].dropna()
    if len(non_na) == 0:
        continue
    mode = non_na.mode(dropna=True)
    fill_val = mode.iloc[0] if len(mode) else "unknown"
    filled[c] = filled[c].fillna(fill_val)

filled.isna().mean().sort_values(ascending=False).head(15)

Ship | IMO Number              0.0
Ship | Name                    0.0
Ship | Ship type               0.0
Ship | Reporting Period        0.0
Ship | Technical efficiency    0.0
Ship | Port of Registry        0.0
Ship | Home Port               0.0
Ship | Ice Class               0.0
Company | IMO Number           0.0
Company | Name                 0.0
DoC | DoC issue date           0.0
DoC | DoC expiry date          0.0
Verifier | Verifier Name       0.0
Verifier | Verifier Address    0.0
Verifier | Verifier City       0.0
dtype: float64

In [ ]:
# Next step preview: schema-driven extraction by top-level header group.
# Since we built columns as "L0 | L1 | L2", we can split by L0 (group name).

def top_group(col: str) -> str:
    return col.split("|")[0].strip() if "|" in col else col.strip()

groups = pd.Series(df.columns).map(top_group).value_counts().head(20)
groups

Annual monitoring results    77
Ship                          8
Verifier                      6
Monitoring methods            3
Company                       2
DoC                           2
Name: count, dtype: int64

In [ ]:
# Example: split out a "Company"-grouped table and link it by imo_number.
company_group_cols = [c for c in df.columns if top_group(c).lower() == "company"]

company_view = df[[IMO_COL] + company_group_cols].copy()
company_view = company_view.rename(columns={IMO_COL: "imo_number"})
company_view["imo_number"] = company_view["imo_number"].astype("Int64").astype(str)
company_view = company_view[company_view["imo_number"].notna() & (company_view["imo_number"] != "<NA>")]
company_view.head(5), company_view.shape

(  imo_number Company | IMO Number                               Company | Name
 3    1013664              5112991               CSL MARITIME S.A.-TOKYO BRANCH
 4    1013676              6409613  BLUEOCEAN SHIPMANAGEMENT INTERNATIONAL INC.
 5    1014591              5949344   FIBER MARINE SHIP MANAGEMENT & TRADING CO.
 6    1014606              6431581                  STELLAR SHIPMANAGEMENT INC.
 7    1014618              6431581                  STELLAR SHIPMANAGEMENT INC.,
 (14132, 3))

In [ ]:
# Save these normalized views (still notebook-stage; later we'll load into DB with a pipeline task).
vessels_path = os.path.join(RAW_OUT_DIR, "vessels.parquet")
company_path = os.path.join(RAW_OUT_DIR, "company_view.parquet")

vessels.to_parquet(vessels_path, index=False)
company_view.to_parquet(company_path, index=False)

vessels_path, company_path

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [18]:
# Print "real" column groups from flattened headers (L0 | L1 | L2)
# Paste the output here and we'll map it to an explicit schema.

cols = list(df.columns) if "df" in globals() else list(df_raw.columns)

def top_group(col: str) -> str:
    return col.split("|")[0].strip() if "|" in col else col.strip()

from collections import defaultdict

by_group = defaultdict(list)
for c in cols:
    by_group[top_group(c)].append(c)

summary = (
    pd.DataFrame({"group": list(by_group.keys()), "n_cols": [len(v) for v in by_group.values()]})
    .sort_values(["n_cols", "group"], ascending=[False, True])
    .reset_index(drop=True)
)

print("=== Column groups (top-level) ===")
print(summary.to_string(index=False))

print("\n=== Example columns per group (first 12) ===")
for g in summary["group"].tolist():
    ex = by_group[g][:12]
    print(f"\n[{g}] (n={len(by_group[g])})")
    for c in ex:
        print(f"- {c}")

=== Column groups (top-level) ===
                    group  n_cols
Annual monitoring results      77
                     Ship       8
                 Verifier       6
       Monitoring methods       3
                  Company       2
                      DoC       2

=== Example columns per group (first 12) ===

[Annual monitoring results] (n=77)
- Annual monitoring results | Fuel consumption | Total fuel consumption [m tonnes]
- Annual monitoring results | Fuel consumption | Fuel consumptions assigned to On laden [m tonnes]
- Annual monitoring results | Fuel consumption | Fuel consumptions assigned to Cargo heating [m tonnes]
- Annual monitoring results | CO₂ Emissions | Total CO₂ emissions [m tonnes]
- Annual monitoring results | CO₂ Emissions | CO₂ emissions from all voyages between ports under a MS jurisdiction [m tonnes]
- Annual monitoring results | CO₂ Emissions | CO₂ emissions from all voyages which departed from ports under a MS jurisdiction [m tonnes]
- Annual monitoring